In [1]:
import torch 
from torch import nn 
from torch.utils.data import DataLoader , Dataset 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import cv2 
import os

In [2]:
path_test  = r"C:\Users\10\Projects\intel-image-classification\seg_test\seg_test"
path_train = r"C:\Users\10\Projects\intel-image-classification\seg_train\seg_train"

In [3]:
class MyDataset(Dataset):
    def __init__(self , path ,  H:int = 100 , W:int = 100):
        super(MyDataset , self).__init__()
        
        self.root = path 
        self.H = H
        self.W = W 
        self.x = []
        self.y = []
        
        self.classes = sorted(os.listdir(path))
        
        for idx , cls in enumerate(self.classes):
            full_path = os.path.join(path , cls)
            for img in os.listdir(full_path):
                Image_path = os.path.join(full_path ,img )
                self.x.append(Image_path)
                self.y.append(idx)
    def __len__(self):
        return len(self.y)
    
    
    def __getitem__(self, index):
        img = self.x[index]
        img = cv2.imread(img)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (self.H , self.W))
        img = torch.from_numpy(img).float() / 255.
        img = img.permute(2, 0, 1)
        label = torch.tensor(self.y[index], dtype=torch.long)

        return img, label

In [4]:
dataset_train = MyDataset(path_train, H=64 , W=64 )
dataset_test = MyDataset(path_test, H=64 , W=64 )

In [5]:
dataloader_train  = DataLoader(dataset_train , batch_size=128 , shuffle=True)
dataloader_test  = DataLoader(dataset_test , batch_size=128 , shuffle=False)

In [6]:
from torchvision.transforms import v2

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel , self ).__init__()
        
        self.train_transform = v2.Compose([
            # Randomly flip image horizontally (left ↔ right)
            # Helps model become invariant to left-right orientation
            v2.RandomHorizontalFlip(p=0.5),

            # Randomly flip image vertically (top ↕ bottom)
            # Useful for some datasets, but can be harmful for natural scenes
            v2.RandomVerticalFlip(p=0.2),

            # Randomly rotate image by up to ±20 degrees
            # Improves robustness to object orientation changes
            v2.RandomRotation(degrees=20),

            # Randomly change image brightness, contrast, saturation, and hue
            # Helps model generalize under different lighting conditions
            v2.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            )
        ])
        
        # Block 1: 32 feature maps
        self.b1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 64x64 -> 32x32
            
         
        )

        # Block 2: 64 feature maps
        self.b2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 32x32 -> 16x16
        )

        # Block 3: 128 feature maps
        self.b3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 16x16 -> 8x8
        )

        # Block 4: 256 feature maps
        self.b4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
      
            nn.MaxPool2d(2, 2)  # 8x8 -> 4x4
        )
         # Block 5: 256 feature maps
        self.b5 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(2, 2)  # 8x8 -> 4x4
        )

        # Global pooling to reduce spatial dimensions
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()

        # Fully connected classifier
        self.fc = nn.Sequential(
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 6)  # 6 classes
        )
        
        
    def forward(self, x):
        # Apply augmentation only in training mode
        if self.training:
            x = self.train_transform(x)

        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        x = self.global_pool(x)
        x = self.flatten(x)

        return self.fc(x)

In [7]:
model = MyModel()
model.state_dict()

OrderedDict([('b1.0.weight',
              tensor([[[[-0.0862,  0.0362,  0.0972],
                        [-0.0865,  0.0835,  0.0693],
                        [-0.0734,  0.0890,  0.0269]],
              
                       [[ 0.1899,  0.0122, -0.0411],
                        [-0.1606, -0.1329,  0.1276],
                        [-0.0278, -0.0358, -0.1851]],
              
                       [[-0.1430, -0.1371, -0.0977],
                        [ 0.0944, -0.1037, -0.0807],
                        [-0.1682,  0.0190, -0.0225]]],
              
              
                      [[[-0.0341,  0.0992, -0.0159],
                        [ 0.0973, -0.0868,  0.0596],
                        [ 0.0562, -0.1054, -0.1486]],
              
                       [[ 0.0379, -0.1423,  0.1870],
                        [-0.0265, -0.0226, -0.1438],
                        [ 0.0430, -0.0908, -0.0023]],
              
                       [[ 0.0808, -0.0963,  0.0403],
                        [ 0

In [8]:
pretrain_model = torch.load("model.pt")
pretrain_model

C:\Users\10\AppData\Local\Temp\ipykernel_3864\2999061291.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrain_model = torch.load("model.pt")


OrderedDict([('b1.0.weight',
              tensor([[[[ 1.1154e-01, -1.0349e-01,  1.7577e-01],
                        [-5.2709e-01,  8.3048e-02, -1.0630e-01],
                        [ 2.7343e-01, -5.8758e-02,  5.1204e-02]],
              
                       [[ 3.5047e-01,  4.4094e-01,  1.1414e-01],
                        [ 6.5274e-02, -1.3591e-01, -2.8722e-01],
                        [-1.7009e-03, -6.7068e-01,  1.1804e-01]],
              
                       [[ 5.3663e-01,  1.9408e-01, -1.6236e-01],
                        [-2.4434e-01, -2.7189e-01, -1.2821e-01],
                        [-2.4035e-01, -1.8242e-01,  5.9256e-01]]],
              
              
                      [[[-5.1623e-02,  7.1021e-02,  1.1318e-01],
                        [-2.9692e-01,  1.4718e-01, -3.6441e-01],
                        [-7.9649e-01, -3.2334e-02,  3.6153e-01]],
              
                       [[-2.5786e-01,  2.0474e-01, -1.5250e-01],
                        [-1.5702e-01,  4.5198e

In [9]:
model.load_state_dict(pretrain_model)

<All keys matched successfully>

In [10]:
model.state_dict()

OrderedDict([('b1.0.weight',
              tensor([[[[ 1.1154e-01, -1.0349e-01,  1.7577e-01],
                        [-5.2709e-01,  8.3048e-02, -1.0630e-01],
                        [ 2.7343e-01, -5.8758e-02,  5.1204e-02]],
              
                       [[ 3.5047e-01,  4.4094e-01,  1.1414e-01],
                        [ 6.5274e-02, -1.3591e-01, -2.8722e-01],
                        [-1.7009e-03, -6.7068e-01,  1.1804e-01]],
              
                       [[ 5.3663e-01,  1.9408e-01, -1.6236e-01],
                        [-2.4434e-01, -2.7189e-01, -1.2821e-01],
                        [-2.4035e-01, -1.8242e-01,  5.9256e-01]]],
              
              
                      [[[-5.1623e-02,  7.1021e-02,  1.1318e-01],
                        [-2.9692e-01,  1.4718e-01, -3.6441e-01],
                        [-7.9649e-01, -3.2334e-02,  3.6153e-01]],
              
                       [[-2.5786e-01,  2.0474e-01, -1.5250e-01],
                        [-1.5702e-01,  4.5198e

In [11]:
device = "cuda" if torch.cuda.is_available() else  "cpu"
device

'cuda'

In [12]:
# Calculate accuracy in percentage
def calculate_accuracy(y_true, y_pred):
    correct = (y_true == y_pred).sum().item()
    return (correct / len(y_true)) * 100

In [13]:
model.eval()

y_train_org = []
y_pred_train =[]


with torch.inference_mode():
    model.to(device=device)
    for x_train , y_train in dataloader_train:
        x_train , y_train = x_train.to(device) , y_train.to(device)
        
        y_train_pred = model(x_train)
        
        y_pred_train.append(torch.argmax(y_train_pred.cpu() , dim=1))
        y_train_org.append(y_train.cpu())
        

In [14]:
y_train = torch.cat(y_train_org)
y_pred = torch.cat(y_pred_train)

In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_train , y_pred ))

              precision    recall  f1-score   support

           0       0.84      0.96      0.90      2191
           1       0.98      0.99      0.98      2271
           2       0.86      0.87      0.87      2404
           3       0.92      0.83      0.87      2512
           4       0.92      0.96      0.94      2274
           5       0.96      0.86      0.91      2382

    accuracy                           0.91     14034
   macro avg       0.91      0.91      0.91     14034
weighted avg       0.91      0.91      0.91     14034



In [18]:
model.eval()

y_test_org = []
y_pred_test =[]


with torch.inference_mode():
    model.to(device=device)
    for x_test , y_test in dataloader_test:
        x_test , y_test = x_test.to(device) , y_test.to(device)
        
        y_test_pred = model(x_test)
        
        y_pred_test.append(torch.argmax(y_test_pred.cpu() , dim=1))
        y_test_org.append(y_test.cpu())
        

y_test = torch.cat(y_train_org)
y_pred_test_ = torch.cat(y_pred_train)

In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test , y_pred_test_ ))

              precision    recall  f1-score   support

           0       0.84      0.96      0.90      2191
           1       0.98      0.99      0.98      2271
           2       0.86      0.87      0.87      2404
           3       0.92      0.83      0.87      2512
           4       0.92      0.96      0.94      2274
           5       0.96      0.86      0.91      2382

    accuracy                           0.91     14034
   macro avg       0.91      0.91      0.91     14034
weighted avg       0.91      0.91      0.91     14034

